# Pipeline Completo de Fine-Tuning LoRA — Scripts 01 → 04

Este notebook executa os scripts individuais em sequência no Google Colab:
1. `01_analyze_dataset.py` — limpeza, split, manifesto
2. `02_finetune_lora.py` — fine-tuning LoRA com Unsloth
3. `03_evaluate_model.py` — avaliação (Token-F1, ROUGE-L, segurança)
4. `04_merge_and_push.py` — merge 16-bit + publicação no HuggingFace Hub

**Resultados de referência (treinamento real no Colab T4):**
- GPU: Tesla T4 (14.5 GB VRAM)
- Dataset bruto: 384.095 exemplos → 358.809 após limpeza
- Total steps: 118 | Epoch: 1 | Batch efetivo: 16
- Training Loss step 50: **1.315176** | Val Loss: **1.238496**
- Training Loss step 100: **1.111915** | Val Loss: **1.151186**
- Training Loss step 118: **1.122613** | Val Loss: **1.147409**
- Training Loss final: **1.3513**
- Tempo estimado: ~15 min no Colab T4
- Modelo publicado: `thallesf1/qwen2.5-3b-medpt-lora`

## 0. Configuração — EDITE AQUI

In [ ]:
# ============================================================
# CONFIGURAÇÕES — EDITE ANTES DE RODAR
# ============================================================

HF_USERNAME  = "thallesf1"            # <-- MUDE ISSO
HF_REPO_NAME = "qwen2.5-3b-medpt-lora"
HF_TOKEN     = ""                     # <-- COLE SEU TOKEN AQUI
REPO_URL     = "https://github.com/elbergaliza/Tech-Challenge-FIAP-FASE3.git"

print(f"Modelo será publicado em: {HF_USERNAME}/{HF_REPO_NAME}")

Modelo será publicado em: thallesf1/qwen2.5-3b-medpt-lora


## 1. Instalar dependências

In [3]:
%%capture
!pip install unsloth datasets trl peft transformers huggingface_hub python-dotenv
!pip install --no-deps bitsandbytes accelerate
print("Dependências instaladas!")

In [4]:
import torch
assert torch.cuda.is_available(), "GPU não disponível! Vá em Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


## 2. Baixar projeto

In [5]:
%cd /content
!rm -rf /content/projeto
!rm -rf /content/projeto
!git clone {REPO_URL} /content/projeto
%cd /content/projeto

/content
Cloning into '/content/projeto'...
remote: Enumerating objects: 432, done.
remote: Counting objects: 100% (226/226), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 432 (delta 77), reused 182 (delta 50), pack-reused 206 (from 1)
Receiving objects: 100% (432/432), 123.20 MiB | 39.17 MiB/s, done.
Resolving deltas: 100% (151/151), done.
/content/projeto


In [6]:
# Remove cache de tokenizers corrompido que veio no repo (symlinks quebrados)
!rm -rf /content/projeto/huggingface_tokenizers_cache
print("Cache corrompido removido — modelo será baixado fresh do HuggingFace Hub.")

Cache corrompido removido — modelo será baixado fresh do HuggingFace Hub.


## PASSO 01 — Análise e Preparação do Dataset

Limpa o dataset MedPT, remove conteúdo promocional, aplica split determinístico
por pergunta normalizada (sem vazamento) e salva o `DatasetDict` + manifesto + relatório.

In [7]:
!python tunning/01_analyze_dataset.py \
    --dataset-path  data/train-00000-of-00001.parquet \
    --output-dir    data/processed/medpt_qa \
    --manifest-path data/processed/medpt_qa_manifest.jsonl \
    --report-path   data/processed/medpt_qa_report.json \
    --train-ratio       0.8 \
    --validation-ratio  0.1 \
    --min-answer-tokens 10 \
    --max-answer-tokens 1000 \
    --seed 42

Generating train split: 384095 examples [00:00, 557583.39 examples/s]
Saving the dataset (1/1 shards): 100% 288257/288257 [00:00<00:00, 1656131.80 examples/s]
Saving the dataset (1/1 shards): 100% 35162/35162 [00:00<00:00, 1510509.62 examples/s]
Saving the dataset (1/1 shards): 100% 35389/35389 [00:00<00:00, 1536740.46 examples/s]
{
  "input_dataset_path": "data/train-00000-of-00001.parquet",
  "total_rows": 384095,
  "kept_rows": 358808,
  "removed_rows": 25287,
  "removed_reasons": {
    "answer_too_short": 10651,
    "promotional_answer": 14636
  },
  "split_counts": {
    "train": 288257,
    "validation": 35162,
    "test": 35389
  },
  "unique_questions_per_split": {
    "train": 138571,
    "validation": 17196,
    "test": 17128
  },
  "config": {
    "seed": 42,
    "train_ratio": 0.8,
    "validation_ratio": 0.1,
    "test_ratio": 0.09999999999999995,
    "min_answer_tokens": 10,
    "max_answer_tokens": 1000
  },
  "top_question_types": [
    [
      "Tratamento",
      13371

## PASSO 02 — Fine-tuning LoRA com Unsloth

Carrega o modelo base Qwen2.5-3B-Instruct em 4-bit, aplica adaptadores LoRA
(r=32, α=64, dropout=0.0) e treina com SFTTrainer.

**Referência:** ~118 steps, ~14 min no T4, loss final ~1.35

In [8]:
!python tunning/02_finetune_lora.py \
    --dataset-dir   data/processed/medpt_qa \
    --base-model-name Qwen/Qwen2.5-3B-Instruct \
    --checkpoint-dir  data/checkpoints/qwen2.5-3b-medpt-lora \
    --final-dir       pre-trained/qwen2.5-3b-medpt-lora \
    --max-seq-length  1024 \
    --max-train-samples 1888 \
    --max-validation-samples 500 \
    --per-device-train-batch-size 2 \
    --per-device-eval-batch-size  1 \
    --gradient-accumulation-steps 8 \
    --learning-rate  2e-4 \
    --num-train-epochs 1 \
    --warmup-ratio   0.03 \
    --logging-steps  10 \
    --save-steps     9999 \
    --eval-steps     50 \
    --lora-dropout   0.0 \
    --seed 42

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Un

## PASSO 03 — Avaliação do Modelo

Gera respostas no split de teste e calcula Token-F1, ROUGE-L e heurísticas de
segurança (cautela, excesso de confiança, categorias de risco clínico).

In [9]:
!python tunning/03_evaluate_model.py \
    --dataset-dir    data/processed/medpt_qa \
    --manifest-path  data/processed/medpt_qa_manifest.jsonl \
    --adapter-dir    pre-trained/qwen2.5-3b-medpt-lora \
    --base-model-name Qwen/Qwen2.5-3B-Instruct \
    --output-dir     data/evaluation/qwen2.5-3b-medpt-lora \
    --max-eval-samples 5 \
    --max-new-tokens   256 \
    --temperature      0.0

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
config.json: 100% 661/661 [00:00<00:00, 2.80MB/s]
model.safetensors.index.json: 35.6kB [00:00, 50.1MB/s]
Fetching 2 files: 100% 2/2 [00:16<00:00,  8.25s/it]
Download complete: 100% 6.17G/6.17G [00:16<00:00, 373MB/s]
Loading weights: 100% 434/434 [00:02<00:00, 200.44it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.15MB/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
{
  "model": {
    "adapter_dir": "pre-trained/qwen2.5-3b-medpt-lora",
    "base_model_name": "Qwen/Qwen2.5-3B-Instruct"
  },
  "dataset": {
    "dataset_dir": "data/processed/medpt_qa",
    "manifest_path": "data/processed/medpt_qa_manifest.jsonl",
    "evaluated_samples": 5
  },
  "metrics": {
    "token_f1_mean": 0.148827692856718,
    "rouge_l_mean": 0.104644025456492,
    "bertscore

## PASSO 04 — Merge + Publicação no HuggingFace Hub

Faz merge do adapter LoRA com o modelo base em 16-bit e publica no Hub.

In [13]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
!free -h | grep Mem
import os
os.environ["HF_TOKEN"] = HF_TOKEN

!python tunning/04_merge_and_push.py \
    --adapter-path pre-trained/qwen2.5-3b-medpt-lora \
    --base-model   Qwen/Qwen2.5-3B-Instruct \
    --repo-id      {HF_USERNAME}/{HF_REPO_NAME} \
    --dtype        float16

Mem:            50Gi       1.4Gi        12Gi       3.0Mi        37Gi        48Gi
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[1/4] Carregando modelo base: Qwen/Qwen2.5-3B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:01<00:00, 336.81it/s]
[2/4] Carregando adapter LoRA: pre-trained/qwen2.5-3b-medpt-lora
[3/4] Fazendo merge dos pesos...
[3/4] Salvando modelo mergeado em: merged-model-qwen2.5-3b-medpt-lora
Writing model shards: 100% 1/1 [00:20<00:00, 20.03s/it]
[4/4] Publicando no HuggingFace Hub: thallesf1/qwen2.5-3b-medpt-lora
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...medpt-lora/tokenizer.json: 100% 11.4M/11.4M [00:00<?, ?B/s]

Processing Files (1 / 1)      :   0% 11.4M/6.18G [00:00<03:49, 26.9MB/s, 57.0MB/s  ]

  ...medpt-lora/tokenizer.json: 100% 11.4

## Diagnóstico — verificação dos artefatos gerados

**Dados de referência (treinamento real T4):**

| Step | Training Loss | Val Loss |
|------|--------------|----------|
| 50   | 1.315176     | 1.238496 |
| 100  | 1.111915     | 1.151186 |
| 118  | 1.122613     | 1.147409 |

In [11]:
import json
from pathlib import Path

# ─── Relatório do script 01 ───
report_path = Path("/content/projeto/data/processed/medpt_qa_report.json")
if report_path.exists():
    report = json.loads(report_path.read_text())
    print("[01] Dataset report:")
    print(f"  total_rows:  {report['total_rows']}")
    print(f"  kept_rows:   {report['kept_rows']}")
    print(f"  split_counts: {report['split_counts']}")
else:
    print("[01] Relatório não encontrado.")

# ─── Metadados do script 02 ───
meta_path = Path("/content/projeto/pre-trained/qwen2.5-3b-medpt-lora/training_metadata.json")
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print("\n[02] Training metadata:")
    print(f"  base_model: {meta.get('base_model_name')}")
    print(f"  lora r/alpha: {meta['lora']['r']}/{meta['lora']['alpha']}")
    print(f"  learning_rate: {meta.get('learning_rate')}")
else:
    print("\n[02] Metadados não encontrados.")

# ─── Relatório do script 03 ───
eval_path = Path("/content/projeto/data/evaluation/qwen2.5-3b-medpt-lora/evaluation_summary.json")
if eval_path.exists():
    ev = json.loads(eval_path.read_text())
    print("\n[03] Evaluation summary:")
    print(f"  evaluated_samples: {ev['dataset']['evaluated_samples']}")
    print(f"  token_f1_mean:     {ev['metrics']['token_f1_mean']:.4f}")
    print(f"  rouge_l_mean:      {ev['metrics']['rouge_l_mean']:.4f}")
    print(f"  caution_rate:      {ev['safety']['caution_rate']:.2%}")
    print(f"  overconfidence:    {ev['safety']['overconfidence_rate']:.2%}")
else:
    print("\n[03] Relatório de avaliação não encontrado.")

# ─── Checklist de arquivos ───
print("\n[Checklist de artefatos]")
artifacts = [
    "/content/projeto/data/processed/medpt_qa",
    "/content/projeto/data/processed/medpt_qa_manifest.jsonl",
    "/content/projeto/data/processed/medpt_qa_report.json",
    "/content/projeto/pre-trained/qwen2.5-3b-medpt-lora/adapter_config.json",
    "/content/projeto/pre-trained/qwen2.5-3b-medpt-lora/training_metadata.json",
    "/content/projeto/data/evaluation/qwen2.5-3b-medpt-lora/evaluation_summary.json",
    "/content/projeto/data/evaluation/qwen2.5-3b-medpt-lora/predictions.jsonl",
]
for p in artifacts:
    exists = Path(p).exists()
    print(f"  {'✅' if exists else '❌'} {p}")

[01] Dataset report:
  total_rows:  384095
  kept_rows:   358808
  split_counts: {'train': 288257, 'validation': 35162, 'test': 35389}

[02] Training metadata:
  base_model: Qwen/Qwen2.5-3B-Instruct
  lora r/alpha: 32/64
  learning_rate: 0.0002

[03] Evaluation summary:
  evaluated_samples: 5
  token_f1_mean:     0.1488
  rouge_l_mean:      0.1046
  caution_rate:      0.00%
  overconfidence:    0.00%

[Checklist de artefatos]
  ✅ /content/projeto/data/processed/medpt_qa
  ✅ /content/projeto/data/processed/medpt_qa_manifest.jsonl
  ✅ /content/projeto/data/processed/medpt_qa_report.json
  ✅ /content/projeto/pre-trained/qwen2.5-3b-medpt-lora/adapter_config.json
  ✅ /content/projeto/pre-trained/qwen2.5-3b-medpt-lora/training_metadata.json
  ✅ /content/projeto/data/evaluation/qwen2.5-3b-medpt-lora/evaluation_summary.json
  ✅ /content/projeto/data/evaluation/qwen2.5-3b-medpt-lora/predictions.jsonl


## Pronto!

Para usar o modelo no projeto, adicione no `.env`:
```
HF_PIPELINE_MODEL=seu-usuario/qwen2.5-3b-medpt-lora
```